In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

from sklearn.metrics import (confusion_matrix, classification_report, roc_auc_score, roc_curve, ConfusionMatrixDisplay)

import joblib

RANDOM_STATE = 42
print("All Imports Successful!")

In [ ]:
x_train = pd.read_csv('../data/processed/x_train.csv')
x_test = pd.read_csv('../data/processed/x_test.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').squeeze()
y_test = pd.read_csv('../data/processed/y_test.csv').squeeze()

print("Shapes Loaded:")
print(f'x_train: {x_train.shape}')
print(f'x_test: {x_test.shape}')
print(f'y_train: {y_train.shape} | Class Balance: {y_train.value_counts().to_dict()}')
print(f'y_test: {y_test.shape} | Class Balance: {y_test.value_counts().to_dict()}')

## Evaluation Strategy

### Why not accuracy?
On our test set having 247 No and 47 Yes, a model predicting all 'No' get:
- Accuracy = 247/294 = 84% (looks great but is useless)
- It catches ZERO employees who will eventually leave

### What we actually measure:

**Confusion Matrix** — The foundation of everything:
|                  | Predicted No | Predicted Yes |
|------------------|-------------|--------------|
| **Actual No**    | TN          | FP           |
| **Actual Yes**   | FN          | TP           |

- **FN (False Negative)** = predicted "stays" but actually leaves → MOST COSTLY for HR
- **FP (False Positive)** = predicted "leaves" but actually stays → wastes retention resources

**Precision** = TP / (TP + FP) → "Of everyone we flagged, how many actually left?"
**Recall**    = TP / (TP + FN) → "Of everyone who left, how many did we catch?"
**F1**        = Harmonic mean of Precision & Recall → single balanced score

**ROC-AUC** — measures how well the model *ranks* leavers above stayers across all thresholds.
- 0.5 = random guessing
- 1.0 = perfect separation
- More than 0.75 is generally considered useful for HR use cases

In [ ]:
def evaluate_model(model, x_test, y_test, model_name='Model'):
    """
    Full evaluation suite for a binary classifier.
    Prints merics and plots confusion matrix + ROC curve.
    Returns a dict of key metrics for easy comparison.
    """

    # Predictions
    y_pred = model.predict(x_test)
    y_pred_prob = model.predict_proba(x_test)[:, 1]

    # Metrics
    report = classification_report(y_test, y_pred, target_names=['No', 'Yes'], output_dict=True)
    roc_auc = roc_auc_score(y_test, y_pred_prob)
    cm = confusion_matrix(y_test, y_pred)

    print(f'\n{'='*55}')
    print(f' {model_name}')
    print(f'\n{'='*55}')
    print(classification_report(y_test, y_pred, target_names=['No', 'Yes']))
    print(f' ROC-AUC Score: {roc_auc:.4f}')

    # Plots
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(model_name, fontsize=14, fontweight='bold')

    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No', 'Yes'])
    disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
    axes[0].set_title('Confusion Matrix')

    # ROC Curve
    fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
    axes[1].plot(fpr, tpr, color='darkorange', lw=2,
                 label=f'ROC Curve (AUC = {roc_auc:.3f})')
    axes[1].plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--', label='Random Guess')
    axes[1].set_xlabel('False Positive Rate')
    axes[1].set_ylabel('True Positive Rate (Recall)')
    axes[1].set_title('ROC Curve')
    axes[1].legend(loc='lower right')

    plt.tight_layout()
    os.makedirs('../outputs/figures', exist_ok=True)
    plt.savefig(f'../outputs/figures/{model_name.replace(" ", "_")}_evaluation.png',
                dpi=150, bbox_inches='tight')
    plt.show()

    # ── Return for comparison table later ─────────────────────────────────
    return {
        'Model'       : model_name,
        'Precision_1' : round(report['Yes']['precision'], 4),
        'Recall_1'    : round(report['Yes']['recall'],    4),
        'F1_1'        : round(report['Yes']['f1-score'],  4),
        'ROC_AUC'     : round(roc_auc, 4)
    }

In [ ]:
# Model 1: Logistic Regression
print('Training Logistic Regression...')

lr_model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight='balanced')
lr_model.fit(x_train, y_train)

print('Training Completed!')
lr_results = evaluate_model(lr_model, x_test, y_test, 'Logistic Regression')

###  Reading the Logistic Regression Results

Focus on the **Yes** row in the classification report:
- **Precision** - Of everyone model flagged as "will leave", 47% actually did.
- **Recall** - Of the 47 people who actually left, model caught 38%.
- **F1** - Balanced score between Precision and Recall.

For HR Applications, **Recall on class 1 (Yes) is the most business-critical metric**. Missing someone who leaves (False Negative) is more costly than a false alarm (False Positive).

Logistic Regression serves as our baseline - all further models must beat this.

In [ ]:
# Model 2: Random Forest
print('Training Random Forest...')
rf_model = RandomForestClassifier(n_estimators=200, max_depth=None, random_state=RANDOM_STATE, n_jobs=-1, class_weight='balanced')
rf_model.fit(x_train, y_train)
print('Training Completed!')

rf_results = evaluate_model(rf_model, x_test, y_test, 'Random Forest')

In [ ]:
# Feature Importance from Random Forest
importances = pd.Series(rf_model.feature_importances_, index=x_train.columns).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
importances.head(20).sort_values().plot(kind='barh', color='steelblue', edgecolor='black', linewidth=0.5)
plt.title('Top 20 Feature Importances from Random Forest', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/figures/random_forest_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 10 Features:')
print(importances.head(10).to_string())

## XGBoost - What makes it different?

Random Forest builds trees in **parallel**, independently.
XGBoost builds trees **sequentially** - each new tree corrects the residual errors of all previous trees. This is called **Gradient Boosting**.

The intuition:
1. Tree 1 makes predictions. It gets some wrong.
2. Tree 2 focuses more on the mistakes Tree 1 made.
3. Tree 3 focuses on what Trees 1+2 still got wrong.
4. ... repeat for `n_estimators` rounds.
5. Final prediction = weighted sum of all trees.

This sequential correction makes XGBoost highly accurate but also more prone to overfitting if not tuned. Key hyperparameters control this:

| Parameter        | Role |
|------------------|------|
| `n_estimators`   | Number of boosting rounds (trees) |
| `max_depth`      | Maximum depth per tree — lower = less overfitting |
| `learning_rate`  | How much each tree contributes — lower = more conservative |
| `scale_pos_weight` | XGBoost's way to handle class imbalance |


In [ ]:
# Model 3: XGBoost 

# Calculate scale_pos_weight (neg:pos) in original training set
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

print(f'scale_pos_weight: {scale_pos_weight:.3f}')

xgb_model = xgb.XGBClassifier(
    n_estimators=300, max_depth=4, 
    learning_rate=0.05, subsample=0.8, 
    colsample_bytree=0.8, scale_pos_weight=scale_pos_weight, 
    use_label_encoder=False, eval_metric='logloss', 
    random_state=RANDOM_STATE, n_jobs=-1
)
xgb_model.fit(x_train, y_train)
print('Training Completed!')

xgb_results = evaluate_model(xgb_model, x_test, y_test, 'XGBoost')

In [ ]:
# Feature Importance from XGBoost

xgb_importances = pd.Series(xgb_model.feature_importances_, index=x_train.columns).sort_values(ascending=False)
plt.figure(figsize=(10, 6))
xgb_importances.head(20).sort_values().plot(kind='barh', color='coral', edgecolor='black', linewidth=0.5)
plt.title('Top 20 Feature Importances from XGBoost', fontsize=14, fontweight='bold')
plt.xlabel('F Score (number of times feature is used in splits)')
plt.tight_layout()
plt.savefig('../outputs/figures/xgboost_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Model Comparison ────────────────────────────────────────────────────────
results_df = pd.DataFrame([lr_results, rf_results, xgb_results])
results_df = results_df.set_index('Model')

print("\n" + "="*58)
print("       FINAL MODEL COMPARISON (Class 1 = Attrition)")
print("="*58)
print(results_df.to_string())
print("="*58)

# Visual comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

results_df[['Precision_1', 'Recall_1', 'F1_1']].plot(
    kind='bar', ax=axes[0], rot=15, colormap='Set2', edgecolor='black', linewidth=0.5
)
axes[0].set_title('Precision / Recall / F1  (Class 1: Yes)')
axes[0].set_ylim(0, 1)
axes[0].legend(loc='upper right')

results_df[['ROC_AUC']].plot(
    kind='bar', ax=axes[1], rot=15, color='steelblue', edgecolor='black', linewidth=0.5, legend=False
)
axes[1].set_title('ROC-AUC Score')
axes[1].set_ylim(0, 1)
axes[1].axhline(y=0.5, color='red', linestyle='--', linewidth=1, label='Random baseline')
axes[1].legend()

plt.tight_layout()
plt.savefig('../outputs/figures/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Save Models

os.makedirs('../models', exist_ok=True)

joblib.dump(lr_model, '../models/Logistic_Regression.pkl')
joblib.dump(rf_model, '../models/Random_Forest.pkl')
joblib.dump(xgb_model, '../models/XGBoost.pkl')

print("All models saved to ../models/")

In [ ]:
# ── Threshold Sensitivity Analysis ─────────────────────────────────────────
from sklearn.metrics import precision_recall_curve

fig, axes = plt.subplots(1, 3, figsize=(15, 6))
models    = [lr_model, rf_model, xgb_model]
names     = ['Logistic Regression', 'Random Forest', 'XGBoost']
colors    = ['steelblue', 'forestgreen', 'darkorange']

for ax, model, name, color in zip(axes, models, names, colors):
    y_prob = model.predict_proba(x_test)[:, 1]
    
    precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob)
    
    ax.plot(thresholds, precisions[:-1], label='Precision', color=color,  lw=2)
    ax.plot(thresholds, recalls[:-1],    label='Recall',    color='black', lw=2, linestyle='--')
    ax.axvline(x=0.5, color='red', linestyle=':', linewidth=1.5, label='Current threshold (0.5)')
    ax.axvline(x=0.3, color='gray', linestyle=':', linewidth=1.5, label='Candidate threshold (0.3)')
    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('Decision Threshold')
    ax.set_ylabel('Score')
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1])
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle('Precision–Recall Tradeoff at Different Thresholds', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/figures/threshold_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()